In [ ]:
# Run the ROM bridge with Smart Agent + TrainerHooks
import sys
sys.path.insert(0, '/content/SCBE-AETHERMOORE/demo')

from rom_emulator_bridge import RomEmulatorBridge
from pathlib import Path

ROM_STEPS = 8000       # ~4 minutes of gameplay
HF_REPO = 'SCBE-AETHER/pokemon-crystal-sft-v1'  # Change to your repo

bridge = RomEmulatorBridge(
    rom_path=Path(ROM_PATH),
    out_dir=Path('/content/SCBE-AETHERMOORE/training-data/rom_sessions'),
    steps=ROM_STEPS,
    sample_every=8,
    hold_ticks=3,
    ocr_every=20,
    max_pairs=1000,
    gif_path=Path('/content/rom_session.gif'),
    gif_fps=10,
    gif_scale=0.5,
    gif_max_frames=300,
    seed=42,
    smart_agent=True,
    game='pokemon_crystal',
)

print(f'ROM: {bridge.rom_title} ({bridge.rom_system})')
print(f'Smart Agent: enabled')
print(f'TrainerHooks: {"enabled" if bridge._hooks_enabled else "disabled"}')
print(f'Steps: {ROM_STEPS}')
print(f'Running...\n')

summary = bridge.run()

print(f'\n=== ROM Bridge Complete ===')
print(f'  ROM:          {summary.rom_title} ({summary.rom_system})')
print(f'  Steps:        {summary.steps}')
print(f'  Training rows:{summary.pairs}')
print(f'  JSONL:        {summary.jsonl_path}')
print(f'  GIF:          {summary.gif_path}')
print(f'  OCR enabled:  {summary.ocr_enabled}')

In [ ]:
# Upload your ROM or mount from Drive
from google.colab import files
import os

ROM_PATH = '/content/pokemon_crystal.gbc'

if not os.path.exists(ROM_PATH):
    print('Upload your Pokemon Crystal ROM (.gbc):')
    uploaded = files.upload()
    for name, data in uploaded.items():
        if name.endswith(('.gbc', '.gb')):
            with open(ROM_PATH, 'wb') as f:
                f.write(data)
            print(f'ROM saved as {ROM_PATH} ({len(data)} bytes)')
            break
    else:
        print('No .gbc/.gb file uploaded. Upload manually or mount from Drive.')
else:
    print(f'ROM found at {ROM_PATH}')

In [ ]:
# Install ROM emulator dependencies
!pip install -q pyboy Pillow pytesseract huggingface_hub
!apt-get -qq install -y tesseract-ocr > /dev/null 2>&1
print('ROM emulator dependencies installed.')
print('NOTE: You must provide your own legally-owned Pokemon Crystal ROM (.gbc)')

# Aethermoor Training Data Factory

Run the Aethermoor Pokemon/Digimon RPG headlessly with AI autopilot to generate
SFT/DPO training data at scale. Push to HuggingFace Hub.

**What this generates:**
- Battle strategy pairs (type effectiveness, spell selection)
- Evolution progression data
- Gacha pull events
- NPC dialogue / knowledge graph conversations
- Player choice SFT pairs
- Dungeon floor clear events

All data is themed around the Six Sacred Tongues (KO/AV/RU/CA/UM/DR)
and the SCBE governance pipeline.

## 1. Setup

In [ ]:
# Install dependencies
!apt-get -qq install -y xvfb > /dev/null 2>&1
!pip install -q pygame-ce numpy pyvirtualdisplay Pillow huggingface_hub python-dotenv
print('Dependencies installed.')

In [ ]:
# Clone repo (or mount from Drive)
import os
if not os.path.exists('/content/SCBE-AETHERMOORE'):
    !git clone https://github.com/issdandavis/scbe-aethermoore-demo.git /content/SCBE-AETHERMOORE
else:
    print('Repo already cloned.')

%cd /content/SCBE-AETHERMOORE

In [ ]:
# Set HuggingFace token (paste yours or use Colab secrets)
import os
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN loaded from Colab secrets.')
except Exception:
    # Manual fallback
    # os.environ['HF_TOKEN'] = 'hf_your_token_here'
    print('Set HF_TOKEN manually if needed.')

## 2. Quick Demo — Single Session (300 steps)

In [ ]:
import sys
sys.path.insert(0, 'demo')
sys.path.insert(0, 'src')

from headless import run_headless

# Quick 300-step demo with AI pilot
hd = run_headless(
    num_steps=300,
    ai_pilot=True,
    save_gif_path='/content/demo_session.gif',
    show_every=100,
    capture_every=4,
    max_frames=200,
    gif_fps=12,
    gif_scale=0.5,
)
print(f'\nStatus: {hd.status()}')

In [ ]:
# Show the GIF inline
from IPython.display import Image, display
if os.path.exists('/content/demo_session.gif'):
    display(Image(filename='/content/demo_session.gif'))
else:
    print('No GIF generated.')

In [ ]:
# Check generated training data
import glob, json

data_files = sorted(glob.glob('training-data/game_sessions/*.jsonl'))
print(f'Training data files: {len(data_files)}')
total_pairs = 0
for f in data_files[-3:]:
    with open(f) as fh:
        pairs = [json.loads(l) for l in fh if l.strip()]
    total_pairs += len(pairs)
    print(f'  {os.path.basename(f)}: {len(pairs)} pairs')
    if pairs:
        print(f'    Sample: {pairs[0]["instruction"][:80]}...')
print(f'\nTotal pairs (recent files): {total_pairs}')

## 3. Scaled Pipeline — Multiple Sessions

In [ ]:
import gc
import time

def run_training_pipeline(
    total_sessions=10,
    steps_per_session=3000,
    strategies=('balanced', 'aggressive', 'cautious'),
    output_dir='/content/training_output',
):
    """Run multiple headless sessions with varied AI strategies."""
    os.makedirs(output_dir, exist_ok=True)
    results = []
    
    for session_idx in range(total_sessions):
        strategy = strategies[session_idx % len(strategies)]
        gif_path = f'{output_dir}/session_{session_idx:03d}_{strategy}.gif'
        
        print(f'\n=== Session {session_idx + 1}/{total_sessions} '
              f'[{strategy}] ({steps_per_session} steps) ===')
        
        t0 = time.time()
        try:
            hd = run_headless(
                num_steps=steps_per_session,
                ai_pilot=True,
                save_gif_path=gif_path,
                show_every=steps_per_session + 1,  # don't show inline (speed)
                capture_every=10,
                max_frames=300,
                gif_fps=10,
                gif_scale=0.4,
            )
            elapsed = time.time() - t0
            results.append({
                'session': session_idx,
                'strategy': strategy,
                'frames': hd.frame_count,
                'elapsed': f'{elapsed:.1f}s',
            })
            print(f'  Done in {elapsed:.1f}s, {hd.frame_count} frames captured')
        except Exception as e:
            print(f'  ERROR: {e}')
            results.append({'session': session_idx, 'error': str(e)})
        
        # Force cleanup between sessions
        gc.collect()
    
    return results

print('Pipeline function defined. Run next cell to start.')

In [ ]:
# Run the scaled pipeline
# Adjust total_sessions and steps_per_session for your needs
# 10 sessions x 3000 steps ~ 15 minutes on Colab

results = run_training_pipeline(
    total_sessions=10,
    steps_per_session=3000,
    strategies=('balanced', 'aggressive', 'cautious'),
)

print('\n=== Pipeline Complete ===')
for r in results:
    print(f'  Session {r.get("session", "?")}: {r}')

## 4. Consolidate & Push to HuggingFace

In [ ]:
import glob, json, random

# Consolidate all JSONL files
all_pairs = []
for path in sorted(glob.glob('training-data/game_sessions/*.jsonl')):
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    all_pairs.append(json.loads(line))
                except json.JSONDecodeError:
                    pass

print(f'Total training pairs: {len(all_pairs)}')

# Category breakdown
categories = {}
for p in all_pairs:
    cat = p.get('metadata', {}).get('category', 'unknown')
    categories[cat] = categories.get(cat, 0) + 1
print('\nCategories:')
for cat, count in sorted(categories.items(), key=lambda x: -x[1]):
    print(f'  {cat}: {count}')

# Train/val split (90/10)
random.shuffle(all_pairs)
split = int(len(all_pairs) * 0.9)
train = all_pairs[:split]
val = all_pairs[split:]

# Write consolidated files
for name, data in [('train', train), ('validation', val)]:
    path = f'/content/aethermoor_{name}.jsonl'
    with open(path, 'w') as f:
        for p in data:
            f.write(json.dumps(p, ensure_ascii=False) + '\n')
    print(f'\nWrote {path}: {len(data)} pairs')

In [ ]:
# Push to HuggingFace Hub
from huggingface_hub import HfApi, create_repo

REPO_ID = 'SCBE-AETHER/aethermoor-sft-v1'  # Change to your repo

api = HfApi()

# Create repo if it doesn't exist
try:
    create_repo(REPO_ID, repo_type='dataset', exist_ok=True)
    print(f'Repo ready: https://huggingface.co/datasets/{REPO_ID}')
except Exception as e:
    print(f'Repo creation: {e}')

# Upload files
for split_name in ['train', 'validation']:
    local_path = f'/content/aethermoor_{split_name}.jsonl'
    if os.path.exists(local_path):
        api.upload_file(
            path_or_fileobj=local_path,
            path_in_repo=f'data/{split_name}.jsonl',
            repo_id=REPO_ID,
            repo_type='dataset',
        )
        print(f'Uploaded {split_name}.jsonl')

print(f'\nDataset live at: https://huggingface.co/datasets/{REPO_ID}')

In [ ]:
# Upload dataset card
DATASET_CARD = """---
language: en
license: apache-2.0
task_categories:
  - text-generation
tags:
  - sft
  - rpg
  - scbe
  - aethermoor
  - sacred-tongues
---

# Aethermoor SFT Training Data

Training data generated by the Aethermoor RPG (SCBE-AETHERMOORE).

## Overview

A Pokemon/Digimon-style RPG generates SFT/DPO training pairs as a
byproduct of gameplay. Every battle, choice, evolution, NPC dialogue,
and dungeon clear produces structured training data.

## Categories

- `game_choice` — Player narrative choices with alternatives
- `battle_training` — Battle actions with type effectiveness
- `evolution_training` — Evolution events with tongue proficiency
- `npc_dialogue` — NPC knowledge graph conversations
- `tower_floor` — Dungeon floor clear events

## Magic System

Six Sacred Tongues with Pokemon-style type effectiveness:
- KO (Kor'aelin) — Authority
- AV (Avali) — Transport
- RU (Runethic) — Policy
- CA (Cassisivadan) — Compute
- UM (Umbroth) — Security
- DR (Draumric) — Schema

## Source

Generated by SCBE-AETHERMOORE game engine with AI autopilot.
"""

with open('/content/README.md', 'w') as f:
    f.write(DATASET_CARD)

api.upload_file(
    path_or_fileobj='/content/README.md',
    path_in_repo='README.md',
    repo_id=REPO_ID,
    repo_type='dataset',
)
print('Dataset card uploaded.')

## 5. Sample Outputs

In [ ]:
# Show sample training pairs
import json

with open('/content/aethermoor_train.jsonl') as f:
    samples = [json.loads(l) for l in f if l.strip()][:10]

for i, s in enumerate(samples):
    print(f'--- Pair {i+1} ---')
    print(f'Category: {s.get("metadata", {}).get("category", "?")}')
    print(f'Instruction: {s["instruction"][:120]}')
    print(f'Response: {s["response"][:120]}')
    print()

## 6. ROM Emulator Pipeline — Pokemon Crystal + AI Training Hooks

Run a legally owned Pokemon Crystal ROM headlessly with the Smart AI Agent.
Every game mechanic generates training data:

- **Pokemon Center** = heal + save checkpoint → full party snapshot as SFT pairs
- **Day Care** = deposit/retrieve → growth/breeding training data
- **Gym** = badge earned → milestone progression data
- **Mart** = purchases → resource management training data
- **Evolution** = species change → progression event data
- **Wild battles** = type effectiveness → battle strategy data
- **Save events** = comprehensive state snapshots → HuggingFace

All data flows through TrainerHooks → JSONL → HuggingFace Hub.